# CME/CBOT Corn Sep 2026 (ZCU26)

This notebook focuses on one real, tradable contract. It reads the
historical OHLCV data from Barchart when the chart is available and uses a
public exact-contract fallback after a Barchart 401/403 or browser-session
failure. It draws
candlesticks and adds causal streaming indicators from Screamer.

The chart uses only actual data for this named contract. It does not stitch
multiple contracts together, so ZCU26 remains the same contract throughout.


In [1]:
from pathlib import Path
import sys

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (candidate for candidate in candidate_roots if (candidate / "Barchart").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this demo from a checkout containing the Barchart folder.")
sys.path.insert(0, str(repo_root))

# The imports intentionally follow the checkout bootstrap above.
# ruff: noqa: E402
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import screamer
from IPython.display import display
from matplotlib.patches import Rectangle
from screamer import ATR, BollingerBands, RollingMean, RollingRSI

from barchart_data import (
    BarchartInteractiveChartError,
    download_history,
    interactive_chart_url,
)

CONTRACT = "ZCU26"
PLOT_SESSIONS = 180
INTERACTIVE_URL = interactive_chart_url(CONTRACT)


## 1. Source page

Running this cell uses one headless Playwright browser session when Barchart
is available. A Barchart 401/403 or notebook browser-session failure
automatically switches to one public exact-contract request, still without
displaying a browser window.


In [2]:
print(f"Official interactive chart: {INTERACTIVE_URL}")


Official interactive chart: https://www.barchart.com/futures/quotes/ZCU26/interactive-chart


## 2. Fetch actual Barchart data

The chart's default range and interval are used when Barchart is available.
The fallback returns the daily history exposed for this exact contract. No
Download control is pressed, no local file is created, and the browser is
closed after the page response has been normalized.


In [3]:
try:
    imported = download_history(CONTRACT, timeout_seconds=120)
except BarchartInteractiveChartError as exc:
    raise RuntimeError(
        "Neither Barchart nor the public exact-contract feed exposed history. "
        "Check the symbol and network, then try again."
    ) from exc

history_source = f"automatic source: {imported.source}"
history = imported.frame
if history.empty:
    raise ValueError(f"No Barchart rows returned for {CONTRACT}.")

history = history.sort_values("date").reset_index(drop=True)
required_columns = {"date", "open", "high", "low", "close", "volume"}
missing_columns = required_columns.difference(history.columns)
if missing_columns:
    raise ValueError(f"Barchart response is missing columns: {sorted(missing_columns)}")
if "openInterest" in history.columns:
    # Barchart can use zero as a missing-value sentinel for the latest row.
    history["openInterest"] = history["openInterest"].where(history["openInterest"] > 0)

close_values = history["close"].to_numpy(dtype=float)
high_values = history["high"].to_numpy(dtype=float)
low_values = history["low"].to_numpy(dtype=float)
volume_values = history["volume"].to_numpy(dtype=float)

bollinger = BollingerBands(window_size=20, num_std=2.0)(close_values)
history[["bb_lower", "bb_mid", "bb_upper"]] = bollinger
history["atr14"] = ATR(window_size=14)(
    high_values,
    low_values,
    close_values,
)
history["rsi14"] = RollingRSI(window_size=14)(close_values)
history["volume_mean20"] = RollingMean(window_size=20)(volume_values)
history["daily_change"] = history["close"].diff()
history["daily_return_pct"] = history["close"].pct_change() * 100

summary = pd.DataFrame(
    {
        "metric": [
            "Contract",
            "Contract month",
            "Indicator engine",
            "First trade date",
            "Last trade date",
            "Rows",
            "Last close (cents/bushel)",
            "Period low",
            "Period high",
            "Latest volume",
        ],
        "value": [
            CONTRACT,
            "September 2026",
            f"Screamer {screamer.__version__}",
            history["date"].min().date(),
            history["date"].max().date(),
            len(history),
            f"{history['close'].iat[-1]:.2f}",
            f"{history['low'].min():.2f}",
            f"{history['high'].max():.2f}",
            f"{int(history['volume'].iat[-1]):,}",
        ],
    }
)
display(summary)


19:46:49 | barchart-data | INFO | Starting history capture: symbol=ZCU26 asset_class=futures headless=True


19:46:49 | barchart-data | ERROR | Browser session could not be started


19:46:49 | barchart-data | WARNING | Barchart browser session was unavailable; using the public exact-contract futures chart feed. This fallback does not bypass Barchart.


19:46:50 | barchart-data | INFO | History captured from public exact-contract feed: rows=688 start=2023-12-14 00:00:00 end=2026-09-11 00:00:00 source=https://query1.finance.yahoo.com/v8/finance/chart/ZCU26.CBT?period1=0&period2=1789321669&interval=1d&events=history&includePrePost=false


,metric,value
0,Contract,ZCU26
1,Contract month,September 2026
2,Indicator engine,Screamer 2.5.0
3,First trade date,2023-12-14
4,Last trade date,2026-09-11
5,Rows,688
6,Last close (cents/bushel),509.50
7,Period low,406.25
8,Period high,526.50
9,Latest volume,200


## 2. Candlesticks with streaming indicators

The chart shows the latest 180 trading sessions for readability. The
indicators are calculated on the full history first, so their warm-up and
rolling state are not restarted at the chart boundary.

- Bollinger Bands: 20-session mean plus or minus two standard deviations.
- ATR: 14-session average true range, a price-volatility measure.
- RSI: 14-session relative strength index.
- Volume mean: 20-session rolling average of volume.

Non-positive open-interest values are treated as missing in the display
because Barchart can use zero as an unavailable-value sentinel. The public
# fallback does not publish open interest, so that column is simply absent.


In [4]:
navy = "#13233A"
gold = "#F4B942"
field_green = "#69B578"
red = "#D95D5D"
sky = "#73B7D8"
muted = "#AEB9C6"

plot_data = history.tail(PLOT_SESSIONS).copy()
x_values = mdates.date2num(
    plot_data["date"].to_numpy(dtype="datetime64[ns]")
)
x_step = np.nanmedian(np.diff(x_values))
candle_width = max(float(x_step * 0.65), 0.25)

plt.style.use("dark_background")
fig, axes = plt.subplots(
    4,
    1,
    figsize=(15, 12),
    sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1, 1.1, 1.1]},
)
fig.patch.set_facecolor(navy)
for axis in axes:
    axis.set_facecolor(navy)
    axis.grid(True, color="#3A4A60", alpha=0.35)
    axis.tick_params(colors=muted)
    for spine in axis.spines.values():
        spine.set_color("#3A4A60")

for x_value, (_, row) in zip(x_values, plot_data.iterrows()):
    bullish = row["close"] >= row["open"]
    color = field_green if bullish else red
    axes[0].vlines(x_value, row["low"], row["high"], color=color, linewidth=0.9)
    body_bottom = min(row["open"], row["close"])
    body_height = abs(row["close"] - row["open"])
    body_height = max(body_height, 0.01)
    axes[0].add_patch(
        Rectangle(
            (x_value - candle_width / 2, body_bottom),
            candle_width,
            body_height,
            facecolor=color,
            edgecolor=color,
            linewidth=0.8,
        )
    )

axes[0].plot(
    x_values,
    plot_data["bb_upper"],
    color=sky,
    linewidth=1.0,
    alpha=0.85,
    label="Bollinger upper",
)
axes[0].plot(
    x_values,
    plot_data["bb_mid"],
    color=gold,
    linewidth=1.0,
    alpha=0.85,
    label="Bollinger mid",
)
axes[0].plot(
    x_values,
    plot_data["bb_lower"],
    color=sky,
    linewidth=1.0,
    alpha=0.85,
    label="Bollinger lower",
)
axes[0].set_title(
    "CBOT Corn | ZCU26 | September 2026",
    loc="left",
    color="white",
    fontsize=16,
    pad=14,
)
axes[0].set_ylabel("Cents/bushel", color=muted)
axes[0].legend(frameon=False, labelcolor="white", loc="upper left", ncol=3)

bar_colors = np.where(
    plot_data["close"].to_numpy() >= plot_data["open"].to_numpy(),
    field_green,
    red,
)
axes[1].bar(
    x_values,
    plot_data["volume"],
    width=candle_width,
    color=bar_colors,
    alpha=0.8,
)
axes[1].plot(
    x_values,
    plot_data["volume_mean20"],
    color=gold,
    linewidth=1.2,
    label="20-session volume mean",
)
axes[1].set_ylabel("Volume", color=muted)
axes[1].legend(frameon=False, labelcolor="white", loc="upper left")

axes[2].axhspan(30, 70, color=sky, alpha=0.08)
axes[2].axhline(70, color=muted, linewidth=0.8, alpha=0.7)
axes[2].axhline(30, color=muted, linewidth=0.8, alpha=0.7)
axes[2].plot(
    x_values,
    plot_data["rsi14"],
    color=sky,
    linewidth=1.5,
    label="RSI(14)",
)
axes[2].set_ylim(0, 100)
axes[2].set_ylabel("RSI", color=muted)
axes[2].legend(frameon=False, labelcolor="white", loc="upper left")

axes[3].plot(
    x_values,
    plot_data["atr14"],
    color=gold,
    linewidth=1.5,
    label="ATR(14)",
)
axes[3].set_ylabel("ATR", color=muted)
axes[3].set_xlabel("Trade date", color=muted)
axes[3].legend(frameon=False, labelcolor="white", loc="upper left")

axes[3].xaxis_date()
axes[3].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
fig.text(
    0.01,
    0.01,
    f"Source: {history_source} | "
    "Screamer indicators | fixed contract: ZCU26",
    color=muted,
    fontsize=9,
)
fig.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()


## 3. Recent observations

This table keeps the latest price move and activity visible without mixing
contracts or applying a roll rule.


In [5]:
recent_columns = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "daily_change",
    "daily_return_pct",
    "atr14",
    "rsi14",
]
if "openInterest" in history.columns:
    recent_columns.insert(6, "openInterest")

recent = history.tail(10)[recent_columns].copy()
recent["date"] = recent["date"].dt.strftime("%Y-%m-%d")
display(recent)


,date,open,high,low,close,volume,daily_change,daily_return_pct,atr14,rsi14
678,2026-08-28,508.00,517.50,507.25,512.00,60581.0,1.75,0.342969,10.854316,75.286305
679,2026-08-31,512.00,518.00,509.00,515.00,5480.0,3.00,0.585938,10.721865,76.152327
680,2026-09-01,514.25,524.75,512.00,521.50,2812.0,6.50,1.262136,10.866731,77.954852
681,2026-09-02,521.75,526.50,514.00,518.75,1359.0,-2.75,-0.527325,10.983393,75.359610
682,2026-09-03,518.00,518.00,503.00,515.25,372.0,-3.50,-0.674699,11.323865,72.070978
683,2026-09-04,516.00,516.00,511.00,512.00,240.0,-3.25,-0.630762,10.872161,69.057371
684,2026-09-08,512.75,515.50,509.75,511.00,261.0,-1.00,-0.195312,10.506292,68.113612
685,2026-09-09,510.75,511.25,505.00,507.75,783.0,-3.25,-0.636008,10.202271,65.004319
686,2026-09-10,508.75,520.75,508.75,514.00,259.0,6.25,1.230921,10.402109,68.027001
687,2026-09-11,512.00,512.00,507.50,509.50,200.0,-4.50,-0.875486,10.123387,63.757045


## Notes

The [Barchart contract page](https://www.barchart.com/futures/quotes/ZCU26/overview)
identifies this instrument as Corn Sep '26 on the CBOT.

The Barchart symbol search documents the contract and month code used by
this fixed contract. This notebook intentionally does not build a nearby or
continuous series; it stays focused on the named ZCU26 contract.

Screamer provides causal rolling indicators with the same API for historical
arrays and live streams. The package is used here for Bollinger Bands, ATR,
RSI, and rolling volume mean.

The package does not store credentials, automate sign-in, click chart
controls, disguise automation, or save the captured response to disk.
